<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Day%201%20-%20LLM%20Fundamentals/Learning/llm_fundamentals_lmstudio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🖥️ LLM Fundamentals — Running a Model Locally with LM Studio

Every other notebook in Day 1 calls a **cloud** LLM — OpenAI, Claude, Gemini, DeepSeek — which means an API key, a bill, and an internet connection. This notebook covers the exact same fundamentals against a model running **entirely on your own machine**, using [LM Studio](https://lmstudio.ai/).

**Why run a model locally?**
- 🔒 **Privacy** — your prompts never leave your machine
- 💰 **No cost** — no API bill, ever
- ✈️ **Works offline** — no internet required once the model is downloaded
- ⚖️ **Trade-off** — quality and speed depend on your hardware; a local model is usually smaller and less capable than a frontier cloud model

> ⚠️ **This notebook will NOT run in Google Colab.** Colab executes in a remote cloud VM that cannot see `localhost` on your laptop. **Run this one locally, in VS Code / Jupyter, on the same machine where LM Studio is running.**

**By the end of this notebook, you will be able to:**
1. Start a local server in LM Studio
2. Connect to it — no API key required
3. Discover which model is actually loaded, instead of guessing its name
4. Make your first call and read the response
5. Control model behavior with sampling parameters
6. Steer the model's behavior with a system instruction
7. Hold a multi-turn conversation using chat history
8. Stream a response token-by-token
9. Read token usage from the response

Follow the cells **in order, top to bottom**.

## ✅ Prerequisites

- [LM Studio](https://lmstudio.ai/) installed on this machine
- At least one model downloaded inside LM Studio (any chat model — pick a small one first, e.g. a 1B–4B parameter model, so it runs fast even on a laptop)
- Python 3.9+ with your bootcamp `venv` activated

## 🚀 Step 1 — Start the Local Server in LM Studio

1. Open **LM Studio**.
2. Go to the **Search** tab and download a chat model if you haven't already (e.g. search "instruct" and pick a small one).
3. Go to the **Developer** tab (the `</>` icon in the left sidebar).
4. Load your downloaded model, then click **Start Server**.
5. Confirm it's running — LM Studio shows a URL, by default: `http://localhost:1234/v1`

LM Studio's server speaks the **same API shape as OpenAI** — so we can reuse the `openai` Python package, just pointed at your own machine instead of OpenAI's servers. No API key is checked; any non-empty string works.

## ⚙️ Step 2 — Install the SDK

In [ ]:
!pip install -q openai

## 🔌 Step 3 — Connect to Your Local Server

If LM Studio is running on a different port, change `BASE_URL` below to match what LM Studio's Developer tab shows.

In [ ]:
from openai import OpenAI

BASE_URL = "http://localhost:1234/v1"

client = OpenAI(base_url=BASE_URL, api_key="lm-studio")  # key is required by the SDK but ignored by LM Studio

print(f"✅ Client configured for {BASE_URL}")

## 🔍 Step 4 — Discover Which Model Is Loaded

Unlike cloud providers, a local model's exact ID depends entirely on what you downloaded — there's nothing to memorize or guess. Ask the server directly, and automatically pick the first chat model (embedding models are excluded).

In [ ]:
models = client.models.list()
chat_models = [m.id for m in models.data if "embed" not in m.id.lower()]

print("Models loaded in LM Studio:")
for model_id in chat_models:
    print(" -", model_id)

if not chat_models:
    raise RuntimeError("No chat model found. Load one in LM Studio's Developer tab and start the server.")

MODEL = chat_models[0]
print(f"\n✅ Using MODEL: {MODEL}")

## 💬 Step 5 — Your First Local LLM Call

Same shape as every cloud provider: text in, text out.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain what a Large Language Model is, in exactly two sentences."}],
    max_tokens=200,
)

print(response.choices[0].message.content)

## 🎛️ Step 6 — Controlling Behavior with Parameters

`temperature` controls randomness — low = focused/deterministic, high = creative/varied. Local models respond to this exactly like their cloud counterparts.

In [ ]:
for temp in [0.0, 1.2]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Give me one creative name for a coffee shop."}],
        temperature=temp,
        max_tokens=100,
    )
    print(f"temperature={temp} -> {response.choices[0].message.content.strip()}")

## 🧭 Step 7 — System Instructions

A **system instruction** sets the model's persona and ground rules before it sees the user's prompt.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a terse senior engineer. Answer in at most 2 lines, no fluff."},
        {"role": "user", "content": "How do I center a div?"},
    ],
    max_tokens=150,
)

print(response.choices[0].message.content)

## 🔄 Step 8 — Multi-Turn Conversations

LLMs are **stateless** — the model has no memory of previous calls. A chat "remembers" only because we resend the full history with every request.

In [ ]:
history = []

def send(user_message: str) -> str:
    history.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(model=MODEL, messages=history, max_tokens=150)
    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

print("Model:", send("My name is Dinesh and I'm building an AI bootcamp."))
print("Model:", send("What did I say I'm building?"))

## ⚡ Step 9 — Streaming Responses

Instead of waiting for the entire response, stream it token-by-token — this is what powers the "typing" effect in chat apps.

In [ ]:
for chunk in client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5, one number per line."}],
    max_tokens=100,
    stream=True,
):
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)

## 🔢 Step 10 — Tokens and Usage

Local models report token usage the same way cloud APIs do — useful for judging how much of the model's context window you're using, even though there's no bill to track.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "The quick brown fox jumps over the lazy dog."}],
    max_tokens=100,
)

print("Prompt tokens:", response.usage.prompt_tokens)
print("Completion tokens:", response.usage.completion_tokens)
print("Total tokens:", response.usage.total_tokens)

## 🎯 Recap

You've now run every core LLM concept from this course against a model that never leaves your machine:

| Concept | What you learned |
|---|---|
| Local hosting | LM Studio exposes an OpenAI-compatible server on `localhost` — no API key, no cost, no internet |
| Model discovery | Ask the server what's loaded (`client.models.list()`) instead of hardcoding a model name |
| Basic call | Same `chat.completions.create()` shape as any cloud OpenAI-compatible provider |
| Sampling controls | `temperature` works exactly as it does in the cloud |
| System instructions | Steering tone and role |
| Multi-turn chat | History re-sent on every call — the model itself has no memory |
| Streaming | Token-by-token delivery, identical to the cloud pattern |
| Tokens & usage | Every call reports prompt/completion/total tokens |

**Next:** compare this to `llm_fundamentals_multi_provider.ipynb` — notice the code is nearly identical to the OpenAI branch there. That's the point of an OpenAI-compatible API: swap `base_url` and you swap providers, cloud or local.